# Modelo juego — versión final compacta

Entrenamiento y comparación de:

- CNN con candidatos legales.
- Bidirectional LSTM.

Los modelos se entrenan con estados parciales y un millón de Sudokus.

El backtracking se integra posteriormente en `resolver_sudoku.py`.

## 1. Conectar Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 2. Importar librerías

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split

print("TensorFlow:", tf.__version__)
print("GPU disponible:", len(tf.config.list_physical_devices("GPU")))

## 3. Configuración

In [ ]:
NUM_SUDOKUS = 1000000
EPOCHS = 15
BATCH_SIZE = 512

# Cambiar a True para entrenar los dos modelos
ENTRENAR_MODELOS = False

## 4. Cargar y preparar los datos

In [ ]:
df = pd.read_csv(
    "/content/drive/MyDrive/proyecto_DL/modelo_juego/data/sudoku.csv",
    nrows=NUM_SUDOKUS,
    dtype=str
)

# df.head()
# df.info()

X = np.array(
    [list(sudoku) for sudoku in df["puzzle"]],
    dtype=np.int8
).reshape(-1, 9, 9)

y = np.array(
    [list(sudoku) for sudoku in df["solution"]],
    dtype=np.int8
).reshape(-1, 9, 9)

print("X:", X.shape)
print("y:", y.shape)

## 5. Separar train y test

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Reservamos una parte del train para validación

X_val = X_train[-50000:]
y_val = y_train[-50000:]

X_train = X_train[:-50000]
y_train = y_train[:-50000]

print("Train:", X_train.shape)
print("Validación:", X_val.shape)
print("Test:", X_test.shape)

## 6. Preparación de estados parciales y candidatos

Los estados parciales simulan tableros que ya han sido completados parcialmente.

Los candidatos legales indican qué números pueden colocarse en cada celda.

El generador crea los batches durante el entrenamiento para no cargar en memoria todas las entradas de 19 canales.

In [ ]:
def crear_estado_parcial(puzzles, soluciones):

    estados = puzzles.copy()

    probabilidad = np.random.uniform(
        0.0,
        0.8,
        size=(len(puzzles), 1, 1)
    )

    revelar = (
        (puzzles == 0)
        & (np.random.random(puzzles.shape) < probabilidad)
    )

    estados[revelar] = soluciones[revelar]

    return estados


def crear_candidatos(tableros):

    one_hot = keras.utils.to_categorical(
        tableros,
        num_classes=10
    ).astype("float32")

    numeros = one_hot[..., 1:]

    usados_fila = numeros.max(axis=2)[:, :, None, :]
    usados_columna = numeros.max(axis=1)[:, None, :, :]

    bloques = numeros.reshape(
        -1, 3, 3, 3, 3, 9
    )

    usados_bloque = bloques.max(axis=(2, 4))
    usados_bloque = np.repeat(
        np.repeat(usados_bloque, 3, axis=1),
        3,
        axis=2
    )

    candidatos = 1 - np.maximum.reduce([
        np.broadcast_to(usados_fila, numeros.shape),
        np.broadcast_to(usados_columna, numeros.shape),
        usados_bloque
    ])

    candidatos[tableros != 0] = 0

    return candidatos.astype("float32")


def crear_entrada_cnn(tableros):

    one_hot = keras.utils.to_categorical(
        tableros,
        num_classes=10
    ).astype("float32")

    candidatos = crear_candidatos(tableros)

    return np.concatenate(
        [one_hot, candidatos],
        axis=-1
    )


class SudokuSequence(keras.utils.Sequence):

    def __init__(
        self,
        puzzles,
        soluciones,
        batch_size,
        tipo_modelo,
        shuffle=True,
        estados_parciales=True
    ):

        super().__init__()

        self.puzzles = puzzles
        self.soluciones = soluciones
        self.batch_size = batch_size
        self.tipo_modelo = tipo_modelo
        self.shuffle = shuffle
        self.estados_parciales = estados_parciales
        self.indices = np.arange(len(puzzles))

        self.on_epoch_end()

    def __len__(self):

        return int(
            np.ceil(
                len(self.puzzles) / self.batch_size
            )
        )

    def __getitem__(self, posicion):

        indices = self.indices[
            posicion * self.batch_size:
            (posicion + 1) * self.batch_size
        ]

        puzzles = self.puzzles[indices]
        soluciones = self.soluciones[indices]

        if self.estados_parciales:
            estados = crear_estado_parcial(
                puzzles,
                soluciones
            )
        else:
            estados = puzzles.copy()

        target = (soluciones - 1).astype("int8")
        pesos = (estados == 0).astype("float32")

        if self.tipo_modelo == "cnn":

            entrada = crear_entrada_cnn(estados)

            return entrada, target, pesos

        entrada = estados.reshape(-1, 81)
        target = target.reshape(-1, 81)
        pesos = pesos.reshape(-1, 81)

        return entrada, target, pesos

    def on_epoch_end(self):

        if self.shuffle:
            np.random.shuffle(self.indices)


train_cnn = SudokuSequence(
    X_train,
    y_train,
    BATCH_SIZE,
    "cnn"
)

val_cnn = SudokuSequence(
    X_val,
    y_val,
    BATCH_SIZE,
    "cnn",
    shuffle=False,
    estados_parciales=False
)

train_lstm = SudokuSequence(
    X_train,
    y_train,
    BATCH_SIZE,
    "lstm"
)

val_lstm = SudokuSequence(
    X_val,
    y_val,
    BATCH_SIZE,
    "lstm",
    shuffle=False,
    estados_parciales=False
)

## 7. Definir la CNN

In [ ]:
modelo_cnn = keras.Sequential([
    keras.Input(shape=(9, 9, 19)),
    layers.Conv2D(128, 3, padding="same", activation="relu"),
    layers.Conv2D(128, 3, padding="same", activation="relu"),
    layers.Conv2D(128, 3, padding="same", activation="relu"),
    layers.Dropout(0.2),
    layers.Conv2D(256, 3, padding="same", activation="relu"),
    layers.Conv2D(256, 3, padding="same", activation="relu"),
    layers.Dropout(0.2),
    layers.Conv2D(9, 1, padding="same", activation="softmax")
])

modelo_cnn.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    weighted_metrics=[
        keras.metrics.SparseCategoricalAccuracy(
            name="accuracy"
        )
    ]
)

modelo_cnn.summary()

## 8. Definir la LSTM

In [ ]:
modelo_lstm = keras.Sequential([
    keras.Input(shape=(81,)),
    layers.Embedding(input_dim=10, output_dim=64),
    layers.Bidirectional(
        layers.LSTM(128, return_sequences=True)
    ),
    layers.Dropout(0.2),
    layers.Bidirectional(
        layers.LSTM(128, return_sequences=True)
    ),
    layers.Dropout(0.2),
    layers.Dense(9, activation="softmax")
])

modelo_lstm.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    weighted_metrics=[
        keras.metrics.SparseCategoricalAccuracy(
            name="accuracy"
        )
    ]
)

modelo_lstm.summary()

## 9. Entrenar los modelos

In [ ]:
if ENTRENAR_MODELOS:

    checkpoint_cnn = keras.callbacks.ModelCheckpoint(
        "/content/drive/MyDrive/proyecto_DL/modelo_juego/modelo/best_cnn_1M.keras",
        monitor="val_accuracy",
        mode="max",
        save_best_only=True
    )

    checkpoint_lstm = keras.callbacks.ModelCheckpoint(
        "/content/drive/MyDrive/proyecto_DL/modelo_juego/modelo/best_lstm_1M.keras",
        monitor="val_accuracy",
        mode="max",
        save_best_only=True
    )

    history_cnn = modelo_cnn.fit(
        train_cnn,
        validation_data=val_cnn,
        epochs=EPOCHS,
        callbacks=[checkpoint_cnn],
        verbose=1
    )

    history_lstm = modelo_lstm.fit(
        train_lstm,
        validation_data=val_lstm,
        epochs=EPOCHS,
        callbacks=[checkpoint_lstm],
        verbose=1
    )

else:

    print(
        "Entrenamiento desactivado. "
        "Cambia ENTRENAR_MODELOS a True para ejecutarlo."
    )

## 10. Evaluar y comparar

La evaluación utiliza una muestra de 10.000 Sudokus.

El modelo ganador se selecciona por mayor accuracy en las celdas vacías.

La resolución final con backtracking se realiza posteriormente en `resolver_sudoku.py`.

In [ ]:
modelo_cnn = keras.models.load_model(
    "/content/drive/MyDrive/proyecto_DL/modelo_juego/modelo/best_cnn_1M.keras"
)

modelo_lstm = keras.models.load_model(
    "/content/drive/MyDrive/proyecto_DL/modelo_juego/modelo/best_lstm_1M.keras"
)

X_evaluacion = X_test[:10000]
y_evaluacion = y_test[:10000]


entrada_cnn = crear_entrada_cnn(
    X_evaluacion
)

pred_cnn = modelo_cnn.predict(
    entrada_cnn,
    batch_size=512,
    verbose=1
)

pred_cnn = np.argmax(
    pred_cnn,
    axis=-1
) + 1


entrada_lstm = X_evaluacion.reshape(
    -1,
    81
)

pred_lstm = modelo_lstm.predict(
    entrada_lstm,
    batch_size=512,
    verbose=1
)

pred_lstm = np.argmax(
    pred_lstm,
    axis=-1
).reshape(-1, 9, 9) + 1


resultado_cnn = np.where(
    X_evaluacion != 0,
    X_evaluacion,
    pred_cnn
)

resultado_lstm = np.where(
    X_evaluacion != 0,
    X_evaluacion,
    pred_lstm
)

In [ ]:
def calcular_metricas(
    puzzles,
    soluciones,
    resultados
):

    vacias = puzzles == 0

    accuracy_vacias = np.mean(
        resultados[vacias]
        == soluciones[vacias]
    )

    sudokus_exactos = np.mean(
        np.all(
            resultados == soluciones,
            axis=(1, 2)
        )
    )

    return accuracy_vacias, sudokus_exactos


accuracy_cnn, exactos_cnn = calcular_metricas(
    X_evaluacion,
    y_evaluacion,
    resultado_cnn
)

accuracy_lstm, exactos_lstm = calcular_metricas(
    X_evaluacion,
    y_evaluacion,
    resultado_lstm
)


comparacion = pd.DataFrame([
    {
        "modelo": "CNN",
        "accuracy_vacias": accuracy_cnn,
        "sudokus_exactos": exactos_cnn
    },
    {
        "modelo": "LSTM",
        "accuracy_vacias": accuracy_lstm,
        "sudokus_exactos": exactos_lstm
    }
])

comparacion

## 11. Guardar el modelo ganador

In [ ]:
if accuracy_cnn >= accuracy_lstm:

    modelo_ganador = modelo_cnn
    tipo_modelo = "cnn"

else:

    modelo_ganador = modelo_lstm
    tipo_modelo = "lstm"


modelo_ganador.save(
    "/content/drive/MyDrive/proyecto_DL/modelo_juego/modelo/best_juego.keras"
)

with open(
    "/content/drive/MyDrive/proyecto_DL/modelo_juego/modelo/tipo_modelo.txt",
    "w"
) as archivo:

    archivo.write(tipo_modelo)


print("Modelo elegido:", tipo_modelo.upper())

## Archivos finales

```text
modelo_juego/modelo/
├── best_cnn_1M.keras
├── best_lstm_1M.keras
├── best_juego.keras
└── tipo_modelo.txt
```